# Network Segregation

## Purpose

This notebook calculates whole-brain network segregation for the retained LEMON participants using the validated Schaefer-200 functional connectivity matrices and the validated seven-network atlas labels.

The analysis compares functional connectivity between ROIs that belong to the **same canonical network** with connectivity between ROIs that belong to **different canonical networks**.

The workflow follows the validated participant, atlas, and FCM data procedures established in the preceding notebooks, while adding the calculations needed to produce one participant-level segregation measure.

## Goal

The goal is to produce, for every retained participant:

1. a pooled mean within-network connectivity value (`within_mean`);
2. a pooled mean between-network connectivity value (`between_mean`); and
3. one whole-brain system segregation score (`system_segregation`).

The analysis will also record the numbers of within-network and between-network edges and the calculation status for each participant so that every retained participant can be accounted for.

The final participant-level measure is:

$$
S = \frac{W-B}{W}
$$

where:

* \(W\) is the mean connectivity across all within-network edges;
* \(B\) is the mean connectivity across all between-network edges; and
* \(S\) is the whole-brain system segregation score.


## 2. Import the Required Tools

This section imports the Python tools needed to load the validated cohort information, functional connectivity data, and Schaefer-200 network labels.

The notebook will reuse the existing functions from the project for participant-to-FCM alignment and reconstruction of connectivity matrices from the validated FCM edge-list files.

Network segregation calculations that are reused across analyses will later be implemented in `src/lemon_connectivity/`, rather than defined only inside this notebook.


In [ ]:
# Import tools for loading validated inputs and calculating network measures
import hashlib
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from lemon_connectivity.alignment import align_participants_to_fcm
from lemon_connectivity.atlas import (
    parse_schaefer_label,
    validate_atlas_annotations,
)
from lemon_connectivity.io import (
    edge_list_to_matrix,
    load_fcm_edge_list,
)
from lemon_connectivity.networks import (
    calculate_network_means,
    calculate_network_statistics,
    calculate_system_segregation,
    create_network_masks,
    preprocess_connectivity_matrix,
)
from lemon_connectivity.qc import batch_quality_control

## 3. Finalize the Connectivity Preprocessing Policy

The network segregation analysis follows the connectivity preprocessing policy finalized in Issue #11.

For each participant, the off-diagonal functional connectivity values will first be transformed using the Fisher \(z\)-transformation:

$$
z = \operatorname{arctanh}(r)
$$

where \(r\) is the original functional connectivity value.

After the Fisher transformation, negative values will be replaced with zero. These zeros remain included in the within-network and between-network means and therefore remain part of their respective denominators.

Diagonal entries will be excluded from all network segregation calculations because they represent self-connections rather than connections between distinct ROIs.

The resulting Fisher-transformed, non-negative edge weights will be used to calculate the within-network mean, between-network mean, and whole-brain system segregation score.


## 4. Load the Validated Inputs

The network segregation calculation requires three validated inputs:

1. the retained-participant inventory;
2. the complete functional connectivity matrices; and
3. the validated Schaefer-200 network-label table.

The retained-participant inventory defines which participants belong in the analysis. The FCM files provide the functional connectivity values for the 200 Schaefer-200 ROIs, while the network-label table identifies the canonical network assigned to each ROI.

These inputs and their established validation procedures are reused from the preceding analysis workflow. The purpose of this section is to load the same validated data sources into this notebook so that the subsequent segregation calculations are performed on the correct participant set and ROI network assignments.

No segregation measure is calculated in this section.


In [ ]:
# Find the repository root so all input paths are defined relative to it
current_path = Path.cwd()

repo_root = next(
    path
    for path in [current_path, *current_path.parents]
    if (path / "pyproject.toml").exists()
)

# Load the validated Schaefer-200 network-label table
atlas_labels_path = repo_root / "data" / "interim" / "schaefer200_7networks_labels.tsv"

atlas_labels = pd.read_csv(
    atlas_labels_path,
    sep="\t",
)
# Create a validation view using the existing Schaefer label parser
atlas_validation = atlas_labels.copy()

atlas_validation["network_id"] = [
    parse_schaefer_label(f"7Networks_{hemisphere}_{parcel_name}")["network_id"]
    for hemisphere, parcel_name in zip(
        atlas_validation["hemisphere"],
        atlas_validation["parcel_name"],
    )
]
# Validate the complete Schaefer-200 atlas using the existing project validator
validate_atlas_annotations(atlas_validation)

### 4.1 Load the Cohort Metadata

The cohort metadata provide the participant identifiers and demographic information used for participant-to-FCM alignment. Final participant eligibility is established by the existing deterministic QC procedure in Section 8.1.

The same validated cohort metadata source used in Notebook 06 is retained here, with participant identifiers loaded as strings so that their formatting is preserved.


In [ ]:
# Load the cohort metadata used for participant-to-FCM alignment
cohort_path = (
    repo_root
    / "data"
    / "external"
    / "Curvature-FCN-Aging"
    / "DATA"
    / "cohort_information.tsv"
)

cohort_metadata = pd.read_csv(
    cohort_path,
    sep="\t",
    dtype={"sub_id": "string"},
)

### 4.2 Define the Canonical Network Order

The segregation analysis uses the seven canonical functional networks defined for the validated Schaefer-200 atlas.

The same ordering established in Notebook 06 is retained here so that network labels and any network-level results are represented consistently throughout the project.

The order is:

Visual → Somatomotor → Dorsal Attention → Salience/Ventral Attention → Limbic → Control → Default


In [ ]:
# Define the canonical network order used throughout the analysis
canonical_network_order = [
    "Visual",
    "Somatomotor",
    "Dorsal Attention",
    "Salience/Ventral Attention",
    "Limbic",
    "Control",
    "Default",
]

### 4.3 Locate and Align the FCM Files

The functional connectivity matrices are stored as FCM edge-list files in the validated external dataset directory.

The cohort metadata and FCM files must be aligned so that each retained participant can be linked to the correct connectivity matrix. We reuse the validated `align_participants_to_fcm()` function from the existing project workflow rather than creating a new participant-matching procedure.


In [ ]:
# Locate the validated FCM directory
fcm_data_path = repo_root / "data" / "external" / "Curvature-FCN-Aging" / "DATA" / "FCM"

# Align cohort participants with their corresponding FCM files
alignment_result = align_participants_to_fcm(
    cohort_metadata,
    fcm_data_path,
)

### 4.4 Inspect the Participant–FCM Alignment

The alignment result records how the cohort participants correspond to the available FCM files.

Before calculating network segregation, we need to confirm that matched participants are available and identify the validated matched-file table that will be used to load the participant-level connectivity matrices.

The alignment table establishes participant-to-FCM matching. Final participant eligibility is determined by the established QC procedure in `Section 8.1`.


In [ ]:
# Inspect the validated participant-to-FCM alignment table
alignment_table = alignment_result.alignment_table

### 4.5 Select the Matched FCM Files

The alignment table identifies FCM files that are successfully matched to cohort participants.

The matched files provide the validated file paths needed for the test-matrix calculation and support the participant-level processing established by the QC procedure in `Section 8.1`.

The files are sorted deterministically so that the processing order is reproducible.

In [ ]:
# Select the validated matched FCM files in a deterministic order
matched_files = alignment_table.loc[
    alignment_table["alignment_status"] == "matched",
    ["matrix_filename", "matrix_path"],
].sort_values(
    ["matrix_filename", "matrix_path"],
    kind="stable",
)

if matched_files.empty:
    raise ValueError("No validated matched FCM files are available")

### 4.6 Inspect the Network Assignments

The Schaefer-200 atlas assigns each of the 200 ROIs to one of seven canonical functional networks.

These network assignments determine how each unique ROI-to-ROI connection will later be classified as either a **within-network** edge or a **between-network** edge.

We inspect the network counts here to confirm that the expected seven-network structure is available before constructing the segregation masks.


In [ ]:
# Count ROIs assigned to each canonical network
network_counts = (
    atlas_labels["canonical_network"].value_counts().reindex(canonical_network_order)
)

### 4.7 Establish the ROI-to-Network Correspondence

The segregation masks depend on the network identity of each matrix position.

The validated atlas uses ROI indices 0–199, matching the 200 ROI positions of the reconstructed FCM matrices. For this analysis, the original validated ROI ordering will be preserved rather than applying a network-based permutation.

This means that each position in the connectivity matrix corresponds directly to the network label at the same ROI index. The same correspondence will be used for every participant.

In [ ]:
# Arrange the network labels in the validated ROI index order
roi_networks = atlas_labels.sort_values("roi_index")["canonical_network"].to_numpy()

## 5. Create the Network Masks

The network masks identify which unique ROI-to-ROI connections belong to the within-network and between-network groups.

For 200 ROIs, there are:

$$
\frac{200(200-1)}{2}=19,900
$$

unique undirected connections.

The analysis will use only the **upper triangle** of the connectivity matrix and exclude the diagonal. This ensures that each connection is counted exactly once and that self-connections are not included.

A connection is classified as:

* **within-network** when both ROIs belong to the same canonical network;
* **between-network** when the two ROIs belong to different canonical networks.

The masks depend only on the ROI network labels, not on the participant's connectivity values. Therefore, the same masks can be applied to every participant's connectivity matrix.

The two masks must be mutually exclusive and together cover all 19,900 unique connections.


In [ ]:
# Calculate the expected number of unique edges in the network groups
expected_total_edges = 200 * (200 - 1) // 2

expected_within_edges = sum(count * (count - 1) // 2 for count in network_counts)

expected_between_edges = expected_total_edges - expected_within_edges

expected_total_edges, expected_within_edges, expected_between_edges

### 5.1 Build the Network Masks

The within-network and between-network masks classify each unique ROI-to-ROI connection according to the network labels of its two ROIs.

Only the upper triangle of the connectivity matrix is used, beginning at \(k=1\). This excludes the diagonal and ensures that each undirected connection is represented exactly once.

The within-network mask identifies pairs whose two ROIs have the same canonical network label. The between-network mask identifies pairs whose network labels differ.

The masks are determined entirely by the ROI network assignments and are therefore identical for every participant.


In [ ]:
# Create the within-network and between-network masks from the validated ROI labels
within_mask, between_mask = create_network_masks(
    roi_networks,
)

### 5.2 Validate the Network Masks

The two masks must satisfy the structural requirements of the segregation analysis.

The within-network and between-network masks must:

* contain exactly the expected number of edges;
* exclude all diagonal entries;
* never classify the same edge as both within-network and between-network; and
* together cover all 19,900 unique ROI-to-ROI connections.

These checks confirm that every unique connection is assigned to exactly one analysis group before connectivity values are introduced.


In [ ]:
# Validate the structure and coverage of the network masks
if within_mask.sum() != expected_within_edges:
    raise ValueError("Within-network mask has an unexpected number of edges")

if between_mask.sum() != expected_between_edges:
    raise ValueError("Between-network mask has an unexpected number of edges")

if np.any(np.diag(within_mask)) or np.any(np.diag(between_mask)):
    raise ValueError("Network masks must exclude the diagonal")

if np.any(within_mask & between_mask):
    raise ValueError("Within- and between-network masks must be disjoint")

if (within_mask | between_mask).sum() != expected_total_edges:
    raise ValueError("Network masks do not cover all unique ROI-to-ROI edges")

within_mask.sum(), between_mask.sum(), (within_mask | between_mask).sum()

## 6. Calculate Connectivity and Network Segregation

The connectivity values must first follow the preprocessing policy established in Section 3.

For each participant, the off-diagonal functional connectivity values will be:

1. transformed from Pearson correlation \(r\) to Fisher \(z\);
2. converted to zero when the resulting value is negative; and
3. used with the previously validated within-network and between-network masks.

The pooled mean within-network connectivity is:

$$
W = \operatorname{mean}(\text{within-network edge weights})
$$

and the pooled mean between-network connectivity is:

$$
B = \operatorname{mean}(\text{between-network edge weights})
$$

Each selected edge contributes equally to its corresponding mean. Therefore, larger networks contribute more within-network edges because they contain more unique ROI pairs.

The whole-brain system segregation score is then:

$$
S = \frac{W-B}{W}
$$

If the pooled within-network mean is at or below the predefined numerical threshold

$$
W \leq 10^{-12}
$$

the segregation score is undefined and will be recorded with an explicit calculation status rather than producing an infinite or arbitrary value.

Negative segregation values are retained when \(B>W\); they are not clipped to zero.


### 6.1 Load a Test Connectivity Matrix

Before applying the network calculations to every retained participant, one validated FCM file is loaded to verify the complete calculation pipeline.

The matrix is reconstructed from its validated edge-list representation using the existing project functions. This single participant is used only as a calculation check at this stage; the full participant-level analysis will be performed later.


In [ ]:
# Load one validated FCM edge list and reconstruct its connectivity matrix
target_file_path = Path(matched_files.iloc[0]["matrix_path"])

fcm_edges = load_fcm_edge_list(
    target_file_path,
    n_rois=200,
)

connectivity_matrix = edge_list_to_matrix(
    fcm_edges,
    n_rois=200,
)

connectivity_matrix.shape

In [ ]:
# Apply the finalized Fisher transformation and negative-edge handling
processed_matrix = preprocess_connectivity_matrix(
    connectivity_matrix,
)

processed_matrix.shape

### 6.2 Calculate the Network Means

The preprocessed connectivity matrix is now combined with the validated within-network and between-network masks.

The pooled within-network mean \(W\) is calculated from all edges classified as within-network, while the pooled between-network mean \(B\) is calculated from all edges classified as between-network.

Because the masks contain only unique upper-triangle edges, each connection contributes exactly once to its corresponding mean.


In [ ]:
# Calculate the pooled within-network and between-network means
within_mean, between_mean = calculate_network_means(
    processed_matrix,
    within_mask,
    between_mask,
)

### 6.3 Calculate System Segregation

Whole-brain system segregation is calculated from the pooled within-network mean \(W\) and pooled between-network mean \(B\):

$$
S = \frac{W-B}{W}
$$

When \(W\) is greater than zero, the score expresses the difference between within-network and between-network connectivity relative to the within-network baseline.

If the pooled within-network mean is zero,

$$
W = 0
$$

the segregation score is undefined. The implementation records an explicit undefined status rather than producing an infinite or arbitrary value.

Negative segregation values are retained when between-network connectivity exceeds within-network connectivity.


In [ ]:
# Calculate the whole-brain system segregation score
system_segregation = calculate_system_segregation(
    within_mean,
    between_mean,
)

## 7. Validate the Calculation with Synthetic Data

Before applying the segregation calculation to all retained participants, the reusable network functions will be tested using small synthetic connectivity matrices with known network assignments and expected results.

The synthetic tests will verify that:

* within-network and between-network edges are classified correctly;
* the diagonal is excluded;
* each undirected edge is counted only once;
* the expected within- and between-network edge counts are recovered;
* the pooled within-network and between-network means are correct;
* the system segregation formula returns the known value;
* a zero within-network mean is handled explicitly;
* the calculation is unchanged when the connectivity matrix and network labels are reordered together; and
* pooled-edge weighting is preserved when networks have unequal sizes.

These checks provide a controlled validation of the reusable implementation before it is applied to participant-level data.


### 7.1 Create a Synthetic Connectivity Matrix

A small four-ROI connectivity matrix is used so that the expected results can be calculated exactly after applying the finalized preprocessing policy.

The four ROIs are assigned to two networks:

* ROIs 0 and 1 → Network A
* ROIs 2 and 3 → Network B

This produces two within-network edges and four between-network edges.

The raw correlation values are chosen so that their Fisher \(z\)-transformed values are known:

* within-network edges: \(z=0.20\) and \(z=0.40\);
* between-network edges: \(z=0.15,\ 0.15,\ 0.15,\ 0.15\).

Because

$$
r=\tanh(z),
$$

the corresponding raw correlations are approximately:

* \(\tanh(0.20)=0.1974\);
* \(\tanh(0.40)=0.3799\);
* \(\tanh(0.15)=0.1489\).

After preprocessing, the expected means are therefore:

$$
W=\frac{0.20+0.40}{2}=0.30
$$

$$
B=\frac{0.15+0.15+0.15+0.15}{4}=0.15
$$

and:

$$
S=\frac{0.30-0.15}{0.30}=0.50.
$$

The original synthetic connectivity matrix contains unit diagonal values, as expected for a correlation matrix. After preprocessing, the working matrix has its diagonal explicitly set to zero. The network masks also exclude the diagonal, ensuring that self-connections do not contribute to the segregation calculation.


In [ ]:
# Create a synthetic connectivity matrix whose Fisher z-values have known answers
synthetic_matrix = np.array(
    [
        [1.0, np.tanh(0.20), np.tanh(0.15), np.tanh(0.15)],
        [np.tanh(0.20), 1.0, np.tanh(0.15), np.tanh(0.15)],
        [np.tanh(0.15), np.tanh(0.15), 1.0, np.tanh(0.40)],
        [np.tanh(0.15), np.tanh(0.15), np.tanh(0.40), 1.0],
    ]
)

synthetic_network_labels = np.array(
    ["Network A", "Network A", "Network B", "Network B"]
)

### 7.2 Apply the Synthetic Network Masks

The same reusable mask function used for the real Schaefer-200 analysis is applied to the synthetic network labels.

With two ROIs in each of two networks, the four-ROI matrix contains:

$$
\frac{4(4-1)}{2}=6
$$

unique undirected edges.

Of these, two are within-network edges and four are between-network edges.

This verifies that the mask logic works independently of the size of the real atlas.


In [ ]:
# Create synthetic within-network and between-network masks
synthetic_within_mask, synthetic_between_mask = create_network_masks(
    synthetic_network_labels,
)

synthetic_within_mask.sum(), synthetic_between_mask.sum()

### 7.3 Calculate the Known Synthetic Means

The synthetic connectivity matrix is processed using the same finalized preprocessing policy used for the real participant matrices.

The Fisher \(z\)-transformation converts the selected raw correlations back to the known values defined in Section 7.1. Because all synthetic correlations are positive, the negative-edge rule does not alter these values.

The reusable network-mean function is then applied to the synthetic matrix and masks.

The expected results are:

$$
W = 0.30
$$

$$
B = 0.15
$$

These known values provide a direct check that the preprocessing and mask-based mean calculations are working correctly.


In [ ]:
# Apply the finalized preprocessing policy to the synthetic matrix
synthetic_processed_matrix = preprocess_connectivity_matrix(
    synthetic_matrix,
)

# Calculate the synthetic within-network and between-network means
synthetic_within_mean, synthetic_between_mean = calculate_network_means(
    synthetic_processed_matrix,
    synthetic_within_mask,
    synthetic_between_mask,
)

### 7.4 Calculate the Known Synthetic Segregation Score

The synthetic within-network and between-network means are now used to calculate the system segregation score with the same reusable function used for participant-level analysis.

From the known values:

$$
W=0.30
$$

$$
B=0.15
$$

the expected segregation score is:

$$
S=\frac{0.30-0.15}{0.30}=0.50.
$$

The calculated value will be compared with this known result using a numerical tolerance to account for floating-point representation.


In [ ]:
# Calculate the synthetic system segregation score
synthetic_segregation = calculate_system_segregation(
    synthetic_within_mean,
    synthetic_between_mean,
)

# Verify that the calculated score matches the known answer
if not np.isclose(synthetic_segregation, 0.50):
    raise ValueError("Synthetic segregation score does not match the expected value")

### 7.5 Validate Edge-Case Handling

The preprocessing and segregation functions must handle two important special cases correctly.

First, negative functional-connectivity values must be replaced by zero after the Fisher transformation. These zeros remain included in the corresponding network mean.

Second, when the pooled within-network mean is zero,

$$
W = 0
$$

the segregation score is undefined. The implementation therefore records an explicit calculation status rather than producing an infinite or arbitrary value.


In [ ]:
# Validate negative-edge handling and zero-denominator handling
edge_case_matrix = np.array(
    [
        [1.0, -0.20, 0.10, 0.10],
        [-0.20, 1.0, 0.10, 0.10],
        [0.10, 0.10, 1.0, -0.20],
        [0.10, 0.10, -0.20, 1.0],
    ]
)

edge_case_processed = preprocess_connectivity_matrix(
    edge_case_matrix,
)

if not np.isclose(edge_case_processed[0, 1], 0.0):
    raise ValueError("Negative connectivity was not replaced with zero")

try:
    calculate_system_segregation(
        0.0,
        0.0,
    )

except ValueError:
    pass
else:
    raise ValueError("Zero within-network mean was not handled explicitly")

### 7.6 Validate Reordering Invariance

The segregation result should not depend on the order in which the ROIs appear in the connectivity matrix, provided that the network labels are reordered using the same ROI permutation.

The synthetic matrix and its network labels will therefore be reordered together. The within-network mean, between-network mean, and system segregation score should remain unchanged.

This verifies that the calculation depends on the connectivity relationships and corresponding network assignments rather than on an arbitrary matrix ordering.


In [ ]:
# Reorder the synthetic matrix and network labels using the same permutation
synthetic_permutation = np.array([2, 0, 3, 1])

reordered_synthetic_matrix = synthetic_matrix[
    np.ix_(synthetic_permutation, synthetic_permutation)
]

reordered_synthetic_labels = synthetic_network_labels[synthetic_permutation]

# Recreate masks using the reordered network labels
reordered_within_mask, reordered_between_mask = create_network_masks(
    reordered_synthetic_labels,
)

# Apply preprocessing and calculate the reordered network means
reordered_processed_matrix = preprocess_connectivity_matrix(
    reordered_synthetic_matrix,
)

reordered_within_mean, reordered_between_mean = calculate_network_means(
    reordered_processed_matrix,
    reordered_within_mask,
    reordered_between_mask,
)

reordered_segregation = calculate_system_segregation(
    reordered_within_mean,
    reordered_between_mean,
)

# Verify invariance under simultaneous matrix and label reordering
if not np.isclose(reordered_within_mean, synthetic_within_mean):
    raise ValueError("Within-network mean changed after simultaneous reordering")

if not np.isclose(reordered_between_mean, synthetic_between_mean):
    raise ValueError("Between-network mean changed after simultaneous reordering")

if not np.isclose(reordered_segregation, synthetic_segregation):
    raise ValueError("System segregation changed after simultaneous reordering")

### 7.7 Validate Pooled-Edge Weighting

The network means are pooled across all selected edges rather than giving each network equal weight.

A five-ROI synthetic example is used with unequal network sizes:

* Network A contains 3 ROIs and therefore has 3 within-network edges.
* Network B contains 2 ROIs and therefore has 1 within-network edge.

The four within-network Fisher \(z\)-values are:

$$
0.10,\ 0.20,\ 0.30,\ 0.80
$$

Therefore, the pooled within-network mean is:

$$
W=\frac{0.10+0.20+0.30+0.80}{4}=0.35
$$

This differs from taking the average of the two network-specific means, which would give \(0.50\). The pooled result confirms that each edge, rather than each network, receives equal weight.

The six between-network edges will each have a Fisher \(z\)-value of \(0.40\), giving:

$$
B=0.40
$$

and therefore a negative segregation value:

$$
S=\frac{0.35-0.40}{0.35}\approx-0.142857.
$$

The negative value is intentionally retained and not clipped to zero.


In [ ]:
# Create an unequal-size synthetic example for pooled-edge weighting
weighted_z_values = np.array(
    [
        [0.0, 0.10, 0.20, 0.40, 0.40],
        [0.10, 0.0, 0.30, 0.40, 0.40],
        [0.20, 0.30, 0.0, 0.40, 0.40],
        [0.40, 0.40, 0.40, 0.0, 0.80],
        [0.40, 0.40, 0.40, 0.80, 0.0],
    ]
)

weighted_matrix = np.tanh(weighted_z_values)
np.fill_diagonal(weighted_matrix, 1.0)

weighted_network_labels = np.array(
    ["Network A", "Network A", "Network A", "Network B", "Network B"]
)

weighted_within_mask, weighted_between_mask = create_network_masks(
    weighted_network_labels,
)

weighted_processed_matrix = preprocess_connectivity_matrix(
    weighted_matrix,
)

weighted_within_mean, weighted_between_mean = calculate_network_means(
    weighted_processed_matrix,
    weighted_within_mask,
    weighted_between_mask,
)

weighted_segregation = calculate_system_segregation(
    weighted_within_mean,
    weighted_between_mean,
)

## 8. Calculate Segregation for All Retained Participants

The retained QC participant inventory defines the participants included in the network segregation analysis.

For each participant in the retained QC inventory, the analysis will:

1. reconstruct the complete \(200 \times 200\) connectivity matrix;
2. apply the finalized Fisher \(z\)-transformation and negative-edge handling;
3. apply the fixed within-network and between-network masks;
4. calculate `within_mean` and `between_mean`;
5. calculate `system_segregation`; and
6. record the participant identifier, edge counts, calculation status, and calculation reason.

The processing will produce one result row for each retained participant. Participant-level results will be kept separate from aggregate summaries so that complete participant accounting can be verified.


### 8.1 Establish the Retained QC Inventory and Participant Processing Table

Participant eligibility for the segregation analysis is taken from the project's existing deterministic FCM quality-control procedure.

The QC table identifies participants retained for analysis using the existing `retained_for_analysis` field. This field incorporates the previously defined filename and content QC checks together with the validated participant-to-FCM alignment status.

Only participants retained by this established QC procedure are carried forward to the participant-level segregation calculation.


In [ ]:
# Run the existing deterministic FCM quality-control procedure
qc_table = batch_quality_control(
    alignment_table,
    fcm_data_path,
    n_rois=200,
)

# Keep only participants retained by the established QC procedure
retained_qc = qc_table.loc[
    qc_table["retained_for_analysis"],
    [
        "canonical_id",
        "matrix_filename",
    ],
].copy()

# Recover the validated FCM paths from the alignment table
retained_qc = retained_qc.merge(
    alignment_table[
        [
            "canonical_id",
            "matrix_filename",
            "matrix_path",
        ]
    ],
    on=["canonical_id", "matrix_filename"],
    how="left",
    validate="one_to_one",
)

# Verify that every retained participant has a validated FCM path
if retained_qc["matrix_path"].isna().any():
    raise ValueError("Retained participants must have a validated matrix_path")

# Verify that retained participants are unique
if retained_qc["canonical_id"].duplicated().any():
    raise ValueError("Retained QC inventory contains duplicate canonical IDs")

participant_files = retained_qc.sort_values(
    ["canonical_id", "matrix_filename"],
    kind="stable",
)

qc_table["retained_for_analysis"].value_counts()

### 8.2 Calculate Segregation for Each Participant

The retained QC participant inventory is processed row by row.

For each participant, the complete FCM edge list is reconstructed into a \(200 \times 200\) connectivity matrix. The reusable network-statistics function then applies the finalized preprocessing policy, fixed within-network and between-network masks, pooled mean calculations, and whole-brain system segregation formula.

The analysis records:

* `canonical_id`;
* `within_mean`;
* `between_mean`;
* `system_segregation`;
* `within_edge_count`;
* `between_edge_count`;
* `calculation_status`; and
* `calculation_reason`.

If the pooled within-network mean is zero,

$$
W = 0
$$

the segregation score is undefined. The participant remains in the output ledger with an explicit calculation status and reason.

Other loading, reconstruction, or calculation failures are also retained in the ledger rather than causing the participant to be silently dropped.


In [ ]:
# Calculate network segregation for every retained QC participant
segregation_results = []

for participant in participant_files.itertuples(index=False):
    result = {
        "canonical_id": participant.canonical_id,
        "within_mean": np.nan,
        "between_mean": np.nan,
        "system_segregation": np.nan,
        "within_edge_count": int(within_mask.sum()),
        "between_edge_count": int(between_mask.sum()),
        "calculation_status": "not_calculated",
        "calculation_reason": "",
    }

    # Load the participant's validated FCM edge list
    try:
        fcm_edges = load_fcm_edge_list(
            Path(participant.matrix_path),
            n_rois=200,
        )
    except (OSError, ValueError) as error:
        result["calculation_status"] = "load_failed"
        result["calculation_reason"] = f"{type(error).__name__}: {error}"
        segregation_results.append(result)
        continue

    # Reconstruct the complete connectivity matrix
    try:
        connectivity_matrix = edge_list_to_matrix(
            fcm_edges,
            n_rois=200,
        )
    except (ValueError, TypeError) as error:
        result["calculation_status"] = "matrix_reconstruction_failed"
        result["calculation_reason"] = f"{type(error).__name__}: {error}"
        segregation_results.append(result)
        continue

    # Calculate all network statistics using the reusable project function
    try:
        statistics = calculate_network_statistics(
            connectivity_matrix,
            roi_networks,
        )
    except ValueError as error:
        error_message = str(error)

        if error_message == (
            "System segregation is undefined because within_mean is zero"
        ):
            result["calculation_status"] = "undefined_within_mean_zero"
        else:
            result["calculation_status"] = "calculation_failed"

        result["calculation_reason"] = f"{type(error).__name__}: {error_message}"
        segregation_results.append(result)
        continue

    result["within_mean"] = statistics["within_mean"]
    result["between_mean"] = statistics["between_mean"]
    result["system_segregation"] = statistics["system_segregation"]
    result["within_edge_count"] = statistics["within_edge_count"]
    result["between_edge_count"] = statistics["between_edge_count"]
    result["calculation_status"] = "calculated"
    result["calculation_reason"] = ""

    segregation_results.append(result)

segregation_results = pd.DataFrame(segregation_results)

segregation_results["calculation_status"].value_counts()

### 8.3 Verify Participant Accounting

The participant-level results must contain exactly one row for every participant in the retained QC inventory.

All participants must have a valid canonical identifier and the expected within-network and between-network edge counts. Calculation status is retained for every participant, including participants whose segregation score is undefined because the within-network mean is zero.

Undefined or failed calculations are therefore retained in the ledger rather than being silently dropped.

In [ ]:
# Verify that the segregation ledger contains exactly the retained QC participants
expected_ids = set(retained_qc["canonical_id"])
actual_ids = set(segregation_results["canonical_id"])

if actual_ids != expected_ids:
    missing_ids = sorted(expected_ids - actual_ids)
    unexpected_ids = sorted(actual_ids - expected_ids)

    raise ValueError(
        f"Participant ledger does not match retained QC inventory. "
        f"Missing IDs: {missing_ids}; "
        f"Unexpected IDs: {unexpected_ids}"
    )

if segregation_results["canonical_id"].duplicated().any():
    raise ValueError("Participant canonical IDs must be unique")

if not (segregation_results["within_edge_count"].eq(expected_within_edges).all()):
    raise ValueError("Unexpected within-network edge count in participant results")

if not (segregation_results["between_edge_count"].eq(expected_between_edges).all()):
    raise ValueError("Unexpected between-network edge count in participant results")

valid_statuses = {
    "calculated",
    "undefined_within_mean_zero",
    "load_failed",
    "matrix_reconstruction_failed",
    "calculation_failed",
}

unexpected_statuses = set(segregation_results["calculation_status"]) - valid_statuses

if unexpected_statuses:
    raise ValueError(f"Unexpected calculation statuses: {sorted(unexpected_statuses)}")

len(expected_ids), len(actual_ids)

## 9. Save the Participant-Level Results

The completed participant-level segregation table is saved as a processed analysis output.

The file contains one row per retained participant and records the canonical participant identifier, within-network mean, between-network mean, whole-brain system segregation score, edge counts, calculation status, and calculation reason.

The processed participant-level output is kept outside the raw and external data directories so that the source connectivity data remain unchanged.

In [ ]:
# Save the participant-level network segregation results
processed_data_path = repo_root / "data" / "processed"

processed_data_path.mkdir(
    parents=True,
    exist_ok=True,
)

segregation_results_path = processed_data_path / "network_segregation_results.tsv"

segregation_results.to_csv(
    segregation_results_path,
    sep="\t",
    index=False,
)


### 9.1 Verify the Saved Results

The saved participant-level output is reloaded from disk and verified against the in-memory results.

The verification confirms that the saved table:

* contains exactly the participants retained by the established QC inventory;
* contains one unique row per retained participant;
* preserves the expected result columns and edge counts;
* preserves the numerical results, including expected missing segregation scores; and
* preserves calculation statuses and calculation reasons.

This verifies that the saved participant-level ledger is an exact representation of the in-memory results, apart from normal floating-point representation and the expected conversion of empty calculation reasons to missing values during file loading.

In [ ]:
# Reload the saved participant-level results for verification
saved_segregation_results = pd.read_csv(
    segregation_results_path,
    sep="\t",
)

required_result_columns = [
    "canonical_id",
    "within_mean",
    "between_mean",
    "system_segregation",
    "within_edge_count",
    "between_edge_count",
    "calculation_status",
    "calculation_reason",
]

if list(saved_segregation_results.columns) != required_result_columns:
    raise ValueError("Saved results contain an unexpected column structure")

# Verify that the saved participant IDs match the retained QC inventory
expected_ids = set(retained_qc["canonical_id"])
saved_ids = set(saved_segregation_results["canonical_id"])

if saved_ids != expected_ids:
    missing_ids = sorted(expected_ids - saved_ids)
    unexpected_ids = sorted(saved_ids - expected_ids)

    raise ValueError(
        f"Saved results do not match the retained QC inventory. "
        f"Missing IDs: {missing_ids}; "
        f"Unexpected IDs: {unexpected_ids}"
    )

# Verify that participant ordering matches the in-memory results
if not saved_segregation_results["canonical_id"].equals(
    segregation_results["canonical_id"]
):
    raise ValueError("Saved canonical IDs do not match the in-memory results")

# Verify participant counts
if len(saved_segregation_results) != len(segregation_results):
    raise ValueError("Saved results contain an unexpected participant count")

# Verify that canonical IDs remain unique
if saved_segregation_results["canonical_id"].duplicated().any():
    raise ValueError("Saved results contain duplicate canonical IDs")

# Verify numerical results, allowing expected NaN values and
# normal floating-point differences
numeric_result_columns = [
    "within_mean",
    "between_mean",
    "system_segregation",
]

for column in numeric_result_columns:
    if not np.allclose(
        saved_segregation_results[column].to_numpy(),
        segregation_results[column].to_numpy(),
        equal_nan=True,
        rtol=1e-10,
        atol=1e-12,
    ):
        raise ValueError(f"Saved values do not match in-memory values for {column}")

# Verify edge counts
for column in [
    "within_edge_count",
    "between_edge_count",
]:
    if not saved_segregation_results[column].equals(segregation_results[column]):
        raise ValueError(f"Saved values do not match in-memory values for {column}")

# Verify calculation statuses
if not saved_segregation_results["calculation_status"].equals(
    segregation_results["calculation_status"]
):
    raise ValueError("Saved calculation statuses do not match the in-memory results")

# Treat empty and missing calculation reasons as equivalent
saved_reasons = saved_segregation_results["calculation_reason"].fillna("").astype(str)

in_memory_reasons = segregation_results["calculation_reason"].fillna("").astype(str)

if not saved_reasons.equals(in_memory_reasons):
    raise ValueError("Saved calculation reasons do not match the in-memory results")

len(saved_segregation_results), len(segregation_results)


### 9.2 Record Analysis Provenance

The participant-level results are accompanied by a separate provenance record describing the connectivity policy, input sources, software environment, and key analysis settings.

The primary segregation definition uses `edge-policy-v1` and `seg_z_positive`.

The provenance record includes the number of retained participants and processed FCM files, the within-network and between-network edge counts, the predefined within-network denominator rule, the negative Fisher-\(z\) handling rule, input checksums, software versions, and the repository revision used for the analysis.

The provenance record is stored separately from the participant-level results so that participant measurements remain distinct from analysis metadata.

In [ ]:
# Calculate a SHA-256 checksum for one input file
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Build a deterministic checksum for all processed FCM inputs
processed_fcm_paths = sorted(Path(path) for path in participant_files["matrix_path"])

fcm_manifest = "\n".join(
    (path.relative_to(repo_root).as_posix() + "\t" + sha256_file(path))
    for path in processed_fcm_paths
)

fcm_manifest_sha256 = hashlib.sha256(fcm_manifest.encode("utf-8")).hexdigest()

# Calculate a checksum for the exact atlas used in the analysis
atlas_sha256 = sha256_file(atlas_labels_path)

# Record the current repository state
try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=repo_root,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except subprocess.CalledProcessError:
    git_commit = "unavailable"

# Record whether the repository has uncommitted changes
git_status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_root,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

working_tree_clean = not bool(git_status)

# Build the provenance record
provenance = pd.DataFrame(
    [
        {
            "edge_policy_version": "edge-policy-v1",
            "segregation_definition": "seg_z_positive",
            "n_rois": 200,
            "n_retained_participants": len(retained_qc),
            "n_processed_fcm_files": len(processed_fcm_paths),
            "within_edge_count": int(expected_within_edges),
            "between_edge_count": int(expected_between_edges),
            "within_mean_denominator_rule": "W == 0 is undefined",
            "negative_fisher_z_handling": (
                "negative values replaced by zero; zeros retained"
            ),
            "fcm_source": "Curvature-FCN-Aging/DATA/FCM",
            "fcm_manifest_sha256": fcm_manifest_sha256,
            "atlas_source": ("data/interim/schaefer200_7networks_labels.tsv"),
            "atlas_sha256": atlas_sha256,
            "python_version": sys.version.split()[0],
            "numpy_version": np.__version__,
            "pandas_version": pd.__version__,
            "platform": platform.platform(),
            "repository_head": git_commit,
            "working_tree_clean": working_tree_clean,
        }
    ]
)

provenance_path = processed_data_path / "network_segregation_provenance.tsv"

provenance.to_csv(
    provenance_path,
    sep="\t",
    index=False,
)

## 10. Aggregate Descriptive Statistics

The participant-level results are summarized descriptively in the notebook.

These summaries describe the distribution of `within_mean`, `between_mean`, and `system_segregation` across the retained participants. They do not replace the participant-level output and do not alter the individual measurements.


In [ ]:
# Summarize the participant-level network segregation measures
descriptive_summary = segregation_results[
    [
        "within_mean",
        "between_mean",
        "system_segregation",
    ]
].describe()

descriptive_summary